# GAT OOF 하이브리드 — 팀 표준(방식 B)
`hybrid-gcn-oof` 브랜치의 `make_oof_emb.py` → `run_hybrid.py`를 GAT에 그대로 적용합니다 (GCN과 같은 절차).

- 블록 5개: 1~7 / 8~14 / 15~21 / 22~28 / 29~34, 회차마다 나머지 블록 라벨로 GAT 학습
- GAT 설정 = 기존 GAT 본 학습: heads 8, lr 0.01, dropout 0.3, weight decay 5e-4, 시드별 에폭 = 본 학습 best_epoch (100, 330, 159, 405, 122)
- 정렬: 빠진 블록의 라벨 없는 노드로 기준 임베딩 `emb_gat_seed{n}.npy`(저장소)에 Procrustes 정렬
- 테스트(35~49): 기준 임베딩 그대로 / XGBoost: 베이스라인 튜닝값 고정

**런타임 > 런타임 유형 변경 > T4 GPU** 후 위에서부터 실행하세요.

In [ ]:
# 1) 저장소(hybrid-gcn-oof 브랜치) + GAT 기준 임베딩(main 브랜치, LFS) + 패키지
!nvidia-smi -L || echo 'GPU 없음 -> 런타임 유형을 T4 GPU로 바꾸세요'
import getpass, os
REPO = 'github.com/HanyangTechAI/2026-2-HAI-ProjectTeam8.git'
if not os.path.exists('/content/2026-2-HAI-ProjectTeam8'):
    tok = getpass.getpass('GitHub 토큰 (public이면 그냥 엔터): ')
    host = f'{tok}@' if tok else ''
    !git clone -q -b hybrid-gcn-oof https://{host}{REPO} /content/2026-2-HAI-ProjectTeam8
    # GAT 기준 임베딩은 main에만 있음 → main을 따로 받아 GAT 임베딩만 LFS로 내려받기
    !apt-get -qq install -y git-lfs > /dev/null
    !GIT_LFS_SKIP_SMUDGE=1 git clone -q --depth 1 -b main https://{host}{REPO} /content/ref_main
    !cd /content/ref_main && git lfs install --local -q && git lfs pull --include="results/emb/emb_gat_seed*"
    !mkdir -p /content/2026-2-HAI-ProjectTeam8/results/emb && cp /content/ref_main/results/emb/emb_gat_seed*.npy /content/2026-2-HAI-ProjectTeam8/results/emb/
%cd /content/2026-2-HAI-ProjectTeam8
# LFS가 실패하면(파일이 133바이트) GitHub 미디어 주소에서 직접 받기
for s in range(5):
    f = f'results/emb/emb_gat_seed{s}.npy'
    if not os.path.exists(f) or os.path.getsize(f) < 1_000_000:
        url = f'https://media.githubusercontent.com/media/HanyangTechAI/2026-2-HAI-ProjectTeam8/main/results/emb/emb_gat_seed{s}.npy'
        t = globals().get('tok', '')
        auth = f'-H "Authorization: token {t}"' if t else ''
        !curl -sfL {auth} -o {f} {url}
!ls -la results/emb/   # emb_gat_seed0~4.npy 가 각각 약 52MB 여야 함 (133바이트면 실패)
!pip install -q torch-geometric "xgboost>=3.4.1,<3.5"

In [ ]:
# 2) Elliptic 데이터 자동 다운로드
from torch_geometric.datasets import EllipticBitcoinDataset
EllipticBitcoinDataset(root='data/elliptic')
!python scripts/verify_data.py data/elliptic

In [ ]:
%%writefile models/gat.py
"""GAT (Veličković et al. 2018).  계획서 4장 공통 사양: 2층 conv + 선형 분류기, 임베딩 64차원.

  python -m models.gat --data-root data/elliptic --seeds 0 1 2 3 4
  python -m models.gat --data-root data/elliptic --seeds 0 --heads 8 --results-dir results/tune/gat_h8   # 튜닝 예

models/gcn.py를 복사해서 conv 층만 GATConv로 바꾼 파일.
  - heads=8 기본(기존 GAT 최종 설정: heads 8, lr 0.01, dropout 0.3), 헤드당 8차원 → concat 64차원.
  - hybrid/make_oof_emb.py는 GAT(in_dim, hidden, dropout)만 넘기므로 이 기본값(8)이 그대로 쓰인다.
  - 헤드 8개로 바꿔도 헤드당 8차원 × 8 = 64차원이라 임베딩 크기는 그대로.
  - GATConv의 dropout 인자 = 어텐션 계수(이웃 연결)에 거는 드롭아웃.
  - --heads는 common/train_gnn.py에 없는 GAT 전용 인자라, 여기서 먼저 읽고 나머지 인자만 main()에 넘긴다
    (공통 코드는 수정하지 않음).
"""
from __future__ import annotations

import argparse
import sys
from pathlib import Path

import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from common.train_gnn import main  # noqa: E402


class GAT(nn.Module):
    def __init__(self, in_dim: int, hidden: int = 64, dropout: float = 0.5, heads: int = 8):
        super().__init__()
        assert hidden % heads == 0, "hidden은 heads로 나누어떨어져야 함"
        per_head = hidden // heads
        self.conv1 = GATConv(in_dim, per_head, heads=heads, dropout=dropout)   # → per_head × heads = 64
        self.conv2 = GATConv(hidden, per_head, heads=heads, dropout=dropout)   # 이 층의 출력 = 임베딩 (64차원)
        self.head = nn.Linear(hidden, 2)
        self.dropout = dropout

    def forward(self, x, edge_index):
        h = F.elu(self.conv1(x, edge_index))
        h = F.dropout(h, p=self.dropout, training=self.training)
        emb = self.conv2(h, edge_index)
        logits = self.head(F.dropout(F.elu(emb), p=self.dropout, training=self.training))
        return logits, emb


if __name__ == "__main__":
    pre = argparse.ArgumentParser(add_help=False)
    pre.add_argument("--heads", type=int, default=8, help="어텐션 헤드 수 (튜닝: 4 또는 8)")
    gat_args, rest = pre.parse_known_args()
    # GAT 기본값: patience 100, 최대 500에폭 (실행할 때 --patience / --max-epochs를 주면 그 값이 우선)
    if not any(a.startswith("--patience") for a in rest):
        rest += ["--patience", "100"]
    if not any(a.startswith("--max-epochs") for a in rest):
        rest += ["--max-epochs", "500"]
    print(f"GAT heads={gat_args.heads}, args={rest}")
    main("gat", lambda in_dim, a: GAT(in_dim, a.hidden, a.dropout, gat_args.heads), argv=rest)


In [ ]:
# 3) 기준 임베딩이 이 GAT 구조(heads 8)와 맞는지 확인: 크기 (203769, 64)여야 함
import numpy as np
for s in range(5):
    e = np.load(f'results/emb/emb_gat_seed{s}.npy'); print(s, e.shape, e.dtype)
# 빠른 점검: 시드 0, 3에폭 (1~2분)
!python -m hybrid.make_oof_emb --model gat --lr 0.01 --dropout 0.3 --weight-decay 5e-4 --seeds 0 --best-epochs 3 --out-dir results_smoke/emb_oof

In [ ]:
# 4) 본 실행 ①: GAT OOF 임베딩 (시드 0~4, 가장 오래 걸림) → results/emb_oof/emb_oof_gat_seed{n}.npy
!python -m hybrid.make_oof_emb --model gat --lr 0.01 --dropout 0.3 --weight-decay 5e-4 --seeds 0 1 2 3 4 --best-epochs 100 330 159 405 122

In [ ]:
# 5) 본 실행 ②: XGBoost 하이브리드 (GCN과 같은 3종)
!python -m hybrid.run_hybrid --mode raw                               # xgb_check (기준선 재현)
!python -m hybrid.run_hybrid --emb-models gat                         # hybrid_gat_naive (누수 있음, 비교용)
!python -m hybrid.run_hybrid --emb-models gat --emb-kind oof          # hybrid_gat (OOF)

In [ ]:
# 6) 결과 요약 -> 이 출력과 폴드 진단표를 Claude에게 붙여 주세요
import pandas as pd
pd.set_option('display.width', 220)
ms = ['xgb_check', 'hybrid_gat_naive', 'hybrid_gat']
df = pd.concat([pd.read_csv(f'results/metrics/{m}.csv', dtype={'threshold_type': str}) for m in ms])
print(df[df.split == 'test'].groupby(['model', 'threshold_type'])[['pr_auc', 'roc_auc', 'precision', 'recall', 'f1']].agg(['mean', 'std']).round(4).to_string())
print(df[df.split != 'test'].pivot_table(index='model', columns='split', values='pr_auc').round(4).to_string())
print(pd.read_csv('results/emb_oof/oof_report_gat.csv').groupby('seed')[['cos_raw', 'cos_aligned_unlabeled', 'cos_aligned_labeled', 'oof_pr_auc_pooled']].mean().round(3).to_string())

In [ ]:
# 7) 드라이브 백업 (gat_oof_results). OOF 임베딩은 git에 안 올리고 드라이브에 둡니다
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/gat_oof_results'
!mkdir -p "$OUT"
!cp -r results/emb_oof results/metrics results/pred "$OUT/"
!cp results/timestep_f1_hybrid_gat*.csv results/timestep_f1_xgb_check.csv results/hybrid_importance_*gat*.csv results/hybrid_importance_xgb_check.csv "$OUT/" 2>/dev/null
!cd /content && zip -qr gat_oof_small.zip "$OUT" -x "*emb_oof/emb_oof_*" && ls -la /content/gat_oof_small.zip
from google.colab import files; files.download('/content/gat_oof_small.zip')   # 임베딩 뺀 작은 zip -> Claude에게 올려 주세요